In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report, accuracy_score, recall_score, precision_score, f1_score, cohen_kappa_score
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, cohen_kappa_score, roc_curve, auc
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

!pip install tensorflow scikit-learn

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
# Update the path to the location of the respective CSV file
#NOTE: This code was developed on google colab so this step might have to be fixed
file_path = '/content/drive/MyDrive/Colab Notebooks/preprocessed_data.csv'

#Dataframe conversion
data = pd.read_csv(file_path, encoding='latin1')

# Define the target and features
X = data.drop(columns=['risk'])
y = data['risk']

# Standardizing the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [29]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ReduceLROnPlateau

from tensorflow.keras.models import Model
from tensorflow.keras import layers, Input

# Define input
input_dim = X_train.shape[1]
input_layer = Input(shape=(input_dim,))

# Encoder
encoded = layers.Dense(128)(input_layer)
encoded = layers.LeakyReLU(alpha=0.1)(encoded)
encoded = layers.Dense(64)(encoded)
encoded = layers.LeakyReLU(alpha=0.1)(encoded)
encoded = layers.Dense(32)(encoded)
encoded = layers.LeakyReLU(alpha=0.1)(encoded)  # Latent space representation

# Decoder
decoded = layers.Dense(64)(encoded)
decoded = layers.LeakyReLU(alpha=0.1)(decoded)
decoded = layers.Dense(128)(decoded)
decoded = layers.LeakyReLU(alpha=0.1)(decoded)
decoded = layers.Dense(input_dim, activation='sigmoid')(decoded)  # Output layer

# Define autoencoder model
autoencoder = Model(input_layer, decoded)
autoencoder.compile(optimizer='adam', loss='mse')

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',  # Monitor validation loss
    factor=0.5,          # Reduce LR by half if no improvement
    patience=5,          # Wait 5 epochs before reducing LR
    min_lr=1e-6,         # Minimum learning rate to avoid extreme reduction
    verbose=1            # Print updates when LR changes
)

# Train the autoencoder
autoencoder.fit(X_train, X_train, epochs=50, batch_size=256, validation_data=(X_test, X_test), verbose=1, callbacks=[lr_scheduler])

# Extract the encoder part (we only need the encoder to transform the features)
encoder = models.Model(inputs=input_layer, outputs=encoded)

# Transform the data using the encoder to get compressed features
X_train_encoded = encoder.predict(X_train)
X_test_encoded = encoder.predict(X_test)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Epoch 1/50
971/971 ━━━━━━━━━━━━━━━━━━━━ 11s 9ms/step - loss: 0.8348 - val_loss: 0.6627 - learning_rate: 0.0010
Epoch 2/50
971/971 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - loss: 0.6667 - val_loss: 0.6605 - learning_rate: 0.0010
Epoch 3/50
971/971 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - loss: 0.7510 - val_loss: 0.6605 - learning_rate: 0.0010
Epoch 4/50
971/971 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 0.6666 - val_loss: 0.6599 - learning_rate: 0.0010
Epoch 5/50
971/971 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - loss: 0.6789 - val_loss: 0.6601 - learning_rate: 0.0010
Epoch 6/50
971/971 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 0.6802 - val_loss: 0.6597 - learning_rate: 0.0010
Epoch 7/50
971/971 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - loss: 0.6513 - val_loss: 0.6591 - learning_rate: 0.0010
Epoch 8/50
971/971 ━━━━━━━━━━━━━━━━━━━━ 13s 10ms/step - loss: 0.6670 - val_loss: 0.6591 - learning_rate: 0.0010
Epoch 9/50
971/971 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - loss: 0.6946 - val_loss: 0.6592 - learning_rate: 0.0010
Epoch 

In [36]:
y_train_np = np.array(y_train)  # Convert to NumPy array

In [38]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score, classification_report

# Define K-Fold Cross-Validation
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42)

cv_f1_scores = []

# Convert y_train to a NumPy array to avoid indexing issues
y_train_np = np.array(y_train)

print("\nPerforming K-Fold Cross-Validation...\n")

# Iterate over each fold
for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X_train_encoded, y_train)):
    print(f"Training Fold {fold_idx + 1} / {kfold.get_n_splits()}")

    # Ensure X_train_encoded is a NumPy array or use .iloc if it's a DataFrame
    if isinstance(X_train_encoded, np.ndarray):
        X_train_fold, X_val_fold = X_train_encoded[train_idx], X_train_encoded[val_idx]
    else:  # If X_train_encoded is a DataFrame
        X_train_fold, X_val_fold = X_train_encoded.iloc[train_idx], X_train_encoded.iloc[val_idx]

    # Use NumPy array for y_train
    y_train_fold, y_val_fold = y_train_np[train_idx], y_train_np[val_idx]

    # Train model on fold
    rf.fit(X_train_fold, y_train_fold)

    # Predict on validation fold
    y_val_pred = rf.predict(X_val_fold)

    # Compute F1 score for the fold
    f1_fold = f1_score(y_val_fold, y_val_pred, average='weighted')
    cv_f1_scores.append(f1_fold)

    print(f"Fold {fold_idx + 1} F1 Score: {f1_fold:.4f}\n")

# Print mean and std of cross-validation scores
print(f"Mean F1 Score from Cross-Validation: {np.mean(cv_f1_scores):.4f} ± {np.std(cv_f1_scores):.4f}")

# Fit final model on full training data
print("\nTraining final model on full dataset...")
rf.fit(X_train_encoded, y_train)
y_pred = rf.predict(X_test_encoded)

# Determine if the problem is binary or multi-class
average_type = 'binary' if len(set(y_train)) == 2 else 'weighted'

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average=average_type)
recall = recall_score(y_test, y_pred, average=average_type)
f1 = f1_score(y_test, y_pred, average=average_type)
kappa = cohen_kappa_score(y_test, y_pred)

# Print individual metrics
print("\nFinal Model Test Set Metrics:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Cohen's Kappa: {kappa:.4f}")




Performing K-Fold Cross-Validation...

Training Fold 1 / 5
Fold 1 F1 Score: 0.8088

Training Fold 2 / 5
Fold 2 F1 Score: 0.8092

Training Fold 3 / 5
Fold 3 F1 Score: 0.8049

Training Fold 4 / 5
Fold 4 F1 Score: 0.8083

Training Fold 5 / 5
Fold 5 F1 Score: 0.8106

Mean F1 Score from Cross-Validation: 0.8083 ± 0.0019

Training final model on full dataset...

Final Model Test Set Metrics:
Accuracy: 0.8212
Precision: 0.8201
Recall: 0.5949
F1 Score: 0.6896
Cohen's Kappa: 0.5684


In [39]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score, classification_report

# Define K-Fold Cross-Validation
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Initialize XGBoost model with regularization and reduced complexity
xgb_model = XGBClassifier(
    eval_metric='logloss',
    random_state=42,
    max_depth=3,  # Limit tree depth to prevent overfitting
    learning_rate=0.1,  # Lower learning rate for better generalization
    n_estimators=100,  # Number of boosting rounds
    reg_alpha=0.1,  # L1 regularization
    reg_lambda=0.1,  # L2 regularization
    scale_pos_weight=np.bincount(y_train)[0] / np.bincount(y_train)[1]  # Balance class weights
)

cv_f1_scores = []

# Convert y_train to a NumPy array to avoid indexing issues
y_train_np = np.array(y_train)

print("\nPerforming K-Fold Cross-Validation...\n")

# Iterate over each fold
for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X_train_encoded, y_train_np)):
    print(f"Training Fold {fold_idx + 1} / {kfold.get_n_splits()}")

    # Ensure X_train_encoded is a NumPy array or use .iloc if it's a DataFrame
    if isinstance(X_train_encoded, np.ndarray):
        X_train_fold, X_val_fold = X_train_encoded[train_idx], X_train_encoded[val_idx]
    else:  # If X_train_encoded is a DataFrame
        X_train_fold, X_val_fold = X_train_encoded.iloc[train_idx], X_train_encoded.iloc[val_idx]

    # Use NumPy array for y_train
    y_train_fold, y_val_fold = y_train_np[train_idx], y_train_np[val_idx]

    # Train XGBoost model on fold
    xgb_model.fit(X_train_fold, y_train_fold)

    # Predict on validation fold
    y_val_pred = xgb_model.predict(X_val_fold)

    # Compute F1 score for the fold
    f1_fold = f1_score(y_val_fold, y_val_pred, average='weighted')
    cv_f1_scores.append(f1_fold)

    print(f"Fold {fold_idx + 1} F1 Score: {f1_fold:.4f}\n")

# Print mean and std of cross-validation scores
print(f"Mean F1 Score from Cross-Validation: {np.mean(cv_f1_scores):.4f} ± {np.std(cv_f1_scores):.4f}")

# Fit final model on full training data
print("\nTraining final model on full dataset...")
xgb_model.fit(X_train_encoded, y_train_np)
y_pred = xgb_model.predict(X_test_encoded)

# Determine if the problem is binary or multi-class
average_type = 'binary' if len(set(y_train)) == 2 else 'weighted'

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average=average_type)
recall = recall_score(y_test, y_pred, average=average_type)
f1 = f1_score(y_test, y_pred, average=average_type)
kappa = cohen_kappa_score(y_test, y_pred)

# Print individual metrics
print("\nFinal Model Test Set Metrics:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Cohen's Kappa: {kappa:.4f}")



Performing K-Fold Cross-Validation...

Training Fold 1 / 5
Fold 1 F1 Score: 0.7944

Training Fold 2 / 5
Fold 2 F1 Score: 0.7952

Training Fold 3 / 5
Fold 3 F1 Score: 0.7897

Training Fold 4 / 5
Fold 4 F1 Score: 0.7918

Training Fold 5 / 5
Fold 5 F1 Score: 0.7940

Mean F1 Score from Cross-Validation: 0.7930 ± 0.0020

Training final model on full dataset...

Final Model Test Set Metrics:
Accuracy: 0.7905
Precision: 0.6508
Recall: 0.8044
F1 Score: 0.7195
Cohen's Kappa: 0.5553


In [41]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers

# Define K-Fold Cross-Validation
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Convert y_train to a NumPy array to avoid indexing issues
y_train_np = np.array(y_train)

cv_f1_scores = []

# Build the DNN model
def build_dnn_model(input_dim):
    model = Sequential()
    model.add(Input(shape=(input_dim,)))  # Specify the input shape
    model.add(Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.01)))
    model.add(Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.01)))
    model.add(Dense(16, activation='relu', kernel_regularizer=regularizers.l2(0.01)))
    model.add(Dense(1, activation='sigmoid'))  # For binary classification, use 'sigmoid'

    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

print("\nPerforming K-Fold Cross-Validation...\n")

# Iterate over each fold
for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X_train_encoded, y_train_np)):
    print(f"Training Fold {fold_idx + 1} / {kfold.get_n_splits()}")

    # Split the data
    X_train_fold, X_val_fold = X_train_encoded[train_idx], X_train_encoded[val_idx]
    y_train_fold, y_val_fold = y_train_np[train_idx], y_train_np[val_idx]

    # Build and train the model
    model = build_dnn_model(X_train_fold.shape[1])
    model.fit(X_train_fold, y_train_fold, epochs=10, batch_size=32, verbose=0)

    # Predict on validation fold
    y_val_pred = (model.predict(X_val_fold) > 0.5).astype(int)

    # Compute F1 score for the fold
    f1_fold = f1_score(y_val_fold, y_val_pred, average='weighted')
    cv_f1_scores.append(f1_fold)

    print(f"Fold {fold_idx + 1} F1 Score: {f1_fold:.4f}\n")

# Print mean and std of cross-validation scores
print(f"Mean F1 Score from Cross-Validation: {np.mean(cv_f1_scores):.4f} ± {np.std(cv_f1_scores):.4f}")

# Train the final model on the entire dataset
print("\nTraining final model on full dataset...")
final_model = build_dnn_model(X_train_encoded.shape[1])
final_model.fit(X_train_encoded, y_train_np, epochs=10, batch_size=32, verbose=1)

# Make predictions on the test set
y_pred = (final_model.predict(X_test_encoded) > 0.5).astype(int)

# Determine if the problem is binary or multi-class
average_type = 'binary' if len(set(y_train)) == 2 else 'weighted'

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average=average_type)
recall = recall_score(y_test, y_pred, average=average_type)
f1 = f1_score(y_test, y_pred, average=average_type)
kappa = cohen_kappa_score(y_test, y_pred)

# Print individual metrics
print("\nFinal Model Test Set Metrics:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Cohen's Kappa: {kappa:.4f}")



Performing K-Fold Cross-Validation...

Training Fold 1 / 5
1553/1553 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
Fold 1 F1 Score: 0.9792

Training Fold 2 / 5
1553/1553 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
Fold 2 F1 Score: 0.9867

Training Fold 3 / 5
1553/1553 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
Fold 3 F1 Score: 0.9895

Training Fold 4 / 5
1553/1553 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
Fold 4 F1 Score: 0.9804

Training Fold 5 / 5
1553/1553 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
Fold 5 F1 Score: 0.9930

Mean F1 Score from Cross-Validation: 0.9858 ± 0.0053

Training final model on full dataset...
Epoch 1/10
7764/7764 ━━━━━━━━━━━━━━━━━━━━ 28s 3ms/step - accuracy: 0.8582 - loss: 0.6514
Epoch 2/10
7764/7764 ━━━━━━━━━━━━━━━━━━━━ 23s 3ms/step - accuracy: 0.9668 - loss: 0.2179
Epoch 3/10
7764/7764 ━━━━━━━━━━━━━━━━━━━━ 40s 3ms/step - accuracy: 0.9755 - loss: 0.1652
Epoch 4/10
7764/7764 ━━━━━━━━━━━━━━━━━━━━ 24s 3ms/step - accuracy: 0.9790 - loss: 0.1399
Epoch 5/10
7764/7764 ━━━━━━━━━━━━━━━━━━━━ 22s 3ms/step - accuracy: 0.9